In [76]:
from pathlib import Path
DATA_DIR = Path.cwd().parent.parent / 'data' / 'interim' 
DATA_DIR
PATH_AI = DATA_DIR / 'ai_all_followers.csv'
PATH_CHATGPT = DATA_DIR / 'chatgpt_all_followers.csv'
PATH_ML = DATA_DIR / 'ml_all_followers.csv'
PATH_OUTPUT = DATA_DIR.parent / 'processed' /'followers.csv'

In [80]:
"""
Script per generare il file 'followers.csv' a partire dai tre file:
  - ai_all_followers.csv
  - chatgpt_all_followers.csv
  - ml_all_followers.csv

Obiettivi:
1. Esattamente 51647 archi (righe).
2. Rapporto archi/nodi ≃ 2.5, con nodi = unici di thread_user_pk + thread_follower_pk.
3. Favorire archi che coinvolgono nodi di alto grado su entrambi i lati.
4. Includere **obbligatoriamente** i seguenti thread_user_pk (da ciascun file):
   - ai:   [63310777769, 63458931542, 63387940542, 70429187510,
            63071917563, 71452042928, 69733929461, 63086999261,
            63400205749, 63895458747, 63258389448, 63464138175,
            68109902162, 63056364516, 66854492977, 63626552478,
            63159194609, 63293944855, 63141700572, 63095622792,
            63861317177, 63077865711, 66378041630, 72016031414,
            66411901180, 66461211257, 63131814221, 63432976554,
            63050637478, 63291410241]
   - chatgpt: [63073067781, 71301727351, 63682150057, 69012907170,
               67998054568, 65352920374, 71806174208, 63451084911, 
               63476330145,69688552123, 71690235692,63452765123, 
               63455512839, 63288603620,66128056965, 63264111449,
               63266800334,70418146574, 63484059496,63081777780, 
               63409959873,63437208949,63055343223, 63517067598,
               63404918397,70698407453,69012907170,68444637261,
               63080302423,72281960423]
   - ml:   [64525247485, 28904998990, 60381420025, 44292826757, 51754083315,
            5760926442, 45700847149, 623659295, 60685382105, 45244372710,
            58751456078, 47680161747, 52692551603, 67071502510, 187264532,
            39643192594, 7468607015, 68951649566, 48929634087, 5472076586,
            6817402894, 71072131998, 1394751896, 50446070550, 6377078357]
"""

import pandas as pd

def main():
    INPUT_PATHS = [
        PATH_AI ,
        PATH_CHATGPT ,
        PATH_ML
    ]
    mandatory_ai = {63310777769, 63458931542, 63387940542, 70429187510,
                    63071917563, 71452042928, 69733929461, 63086999261,
                    63400205749, 63895458747, 63258389448, 63464138175,
                    68109902162, 63056364516, 66854492977, 63626552478,
                    63159194609, 63293944855, 63141700572, 63095622792,
                    63861317177, 63077865711, 66378041630, 72016031414,
                    66411901180, 66461211257, 63131814221, 63432976554,
                    63050637478, 63291410241}
    mandatory_chatgpt = {63073067781, 71301727351, 63682150057, 69012907170,
                         67998054568, 65352920374, 71806174208, 63451084911, 
                         63476330145,69688552123, 71690235692,63452765123, 
                         63455512839, 63288603620,66128056965, 63264111449,
                         63266800334,70418146574, 63484059496,63081777780, 
                         63409959873,63437208949,63055343223, 63517067598,
                         63404918397,70698407453,69012907170,68444637261,
                        63080302423,72281960423}
    mandatory_ml = {64525247485, 28904998990, 60381420025, 44292826757,
                    51754083315, 5760926442, 45700847149, 623659295,
                    60685382105, 45244372710, 58751456078, 47680161747,
                    52692551603, 67071502510, 187264532, 39643192594,
                    7468607015, 68951649566, 48929634087, 5472076586,
                    6817402894, 71072131998, 1394751896, 50446070550,
                    6377078357}
    mandatory_nodes = mandatory_ai | mandatory_chatgpt | mandatory_ml
    
    # --- Parametri ---
    TARGET_EDGES = 50000
    MAX_RATIO    = 2.5
    RAND         = 42
    
    # -- 1) Carico e concateno --
    dfs = [pd.read_csv(p) for p in INPUT_PATHS]
    df = pd.concat(dfs, ignore_index=True)
    # Rimuovo eventuali colonne "count"
    df = df.drop(columns=[c for c in df.columns if "count" in c.lower()], errors="ignore")

    # Assumo questi nomi di colonna
    U = "thread_user_pk"
    F = "thread_follower_pk"

    # --- 2) Calcolo grado di ogni nodo ---
    deg_u = df[U].value_counts().rename("deg_u")
    deg_f = df[F].value_counts().rename("deg_f")
    df = df.merge(deg_u, left_on=U, right_index=True)
    df = df.merge(deg_f, left_on=F, right_index=True)
    df["weight"] = df["deg_u"] + df["deg_f"]

    # --- 3) Seleziono 1 arco per ogni nodo mandatorio (se presente) ---
    mandatory_idxs = set()
    for node in mandatory_nodes:
        # prendo tutti gli indici in cui appare come thread_user_pk
        idxs = df.index[df[U] == node].tolist()
        if not idxs:
            continue
        # scelgo quello di massimo weight
        best = df.loc[idxs, "weight"].idxmax()
        mandatory_idxs.add(best)

    mand_df = df.loc[sorted(mandatory_idxs)]
    print(f"Inclusi {len(mandatory_idxs)} archi obbligatori (nodi trovati: {len(mandatory_idxs)})")

    # --- 4) Greedy sui restanti per ratio edges/nodes ≤ 2.5 ---
    rest_df = df.drop(index=mandatory_idxs)
    rest_sorted = rest_df.sort_values("weight", ascending=False)

    selected = mand_df.to_dict("records")
    nodes = set(mand_df[U]) | set(mand_df[F])

    for _, row in rest_sorted.iterrows():
        if len(selected) >= TARGET_EDGES:
            break
        u,v = row[U], row[F]
        new_nodes = {u,v} - nodes
        e_new = len(selected) + 1
        n_new = len(nodes) + len(new_nodes)
        if e_new / n_new <= MAX_RATIO:
            selected.append(row)
            nodes |= {u,v}

    # --- 5) Completo casuale se necessario ---
    if len(selected) < TARGET_EDGES:
        need = TARGET_EDGES - len(selected)
        fill = rest_sorted.sample(n=need, random_state=RAND)
        selected.extend(fill.to_dict("records"))

    # --- 6) Salvo e stampo check finale ---
    out_df = pd.DataFrame(selected).sample(frac=1, random_state=RAND)
    total_edges = len(out_df)
    total_nodes = out_df[U].nunique() + out_df[F].nunique()
    print(f"Edges: {total_edges}, Nodes: {total_nodes}, Ratio: {total_edges/total_nodes:.3f}")

    out_df.drop(columns=["deg_u","deg_f","weight"], inplace=True)
    out_df.to_csv(PATH_OUTPUT, index=False)
    print(f"👉 '{PATH_OUTPUT}' creato con successo.")

if __name__ == "__main__":
    main()

Inclusi 84 archi obbligatori (nodi trovati: 84)
Edges: 50000, Nodes: 22910, Ratio: 2.182
👉 'c:\Users\pasqu\Desktop\progettoasnm\Code\data_extraction\data\processed\followers.csv' creato con successo.
